# ♟️ Chess Elo Predictor — Dokumentace ML modelu

Tento notebook dokumentuje kompletní proces vývoje modelu strojového učení, který predikuje vývoj šachového ratingu (Elo) hráčů na platformě **Chess.com**.

## Obsah
1. Instalace knihoven
2. Původ a sběr dat (crawling z Chess.com API)
3. Načtení a průzkum dat (EDA)
4. Čištění a předzpracování dat
5. Feature Engineering (tvorba atributů)
6. Rozdělení do Elo pásem (Stratifikace)
7. Trénink LightGBM modelu
8. Vyhodnocení modelu (Evaluace)
9. Vizualizace — Feature Importance
10. Srovnání s K-NN (proč je LightGBM lepší)

> **Zadání:** Model musí být natrénován výhradně na reálných, vlastnoručně sesbíraných datech. Data pocházejí z veřejného Chess.com API a byla stažena vlastním asynchronním crawlerem (`downloader.py`).

## 1. Instalace knihoven

In [ ]:
!pip install pandas numpy lightgbm scikit-learn matplotlib seaborn

## 2. Původ dat — jak jsme data sesbírali

Data pochází z **veřejného REST API platformy Chess.com** (https://www.chess.com/news/view/published-data-api).

Napsali jsme vlastní asynchronní crawler (`downloader.py`), který:
- Začíná u hráčů z populárních klubů
- Stahuje měsíční herní archivy pro každého hráče (JSON)
- Z archivů rekonstruuje **měsíční snapshot** — Elo na konci každého měsíce + počet her
- Ukládá do lokální SQLite databáze (`elo_history_v3.db`)
- Přidává oponenty každého hráče do fronty (BFS — Breadth First Search)
- Filtruje hráče mimo rozsah **600–2000 Elo** (mimo rozsah trénování)

### Proč měsíční snapshoty?
Chess.com nemá endpoint pro historii ratingu — data se musí rekonstruovat z herních archivů. Místo stahování každé jednotlivé hry (gigabajty dat) ukládáme pouze shrnutí za každý měsíc. Tím získáme 100× méně dat při stejné informační hodnotě.

**Celkem sesbíráno: přes 1 500 hráčů s min. 3 měsíci herní historie.**

## 3. Načtení dat z databáze

⚠️ **Před spuštěním:** Nahrajte soubor `elo_history_v3.db` do Google Colab.
- Klikněte na ikonu složky vlevo → **Nahrát do úložiště relace**
- Nebo: `from google.colab import files; files.upload()`

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Připojení k databázi
conn = sqlite3.connect('elo_history_v3.db')

# Načtení všech snapshotů
df_raw = pd.read_sql_query(
    'SELECT username, month_idx, elo, games_count, time_class FROM snapshots',
    conn
)
conn.close()

print(f'✅ Načteno {len(df_raw):,} snapshotů')
print(f'   Unikátních hráčů: {df_raw["username"].nunique():,}')
print(f'   Časová kontrola: {df_raw["time_class"].value_counts().to_dict()}')
df_raw.head(10)

## 4. Průzkum dat (EDA — Exploratory Data Analysis)

In [ ]:
# Základní statistiky
print('=== Statistiky Elo ratingů ===')
print(df_raw['elo'].describe())

# Distribuce Elo ratingů
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df_raw['elo'], bins=50, color='#7c3aed', alpha=0.8)
axes[0].set_title('Distribuce Elo ratingů v datasetu')
axes[0].set_xlabel('Elo rating')
axes[0].set_ylabel('Počet snapshotů')

# Počet snapshotů na hráče
snaps_per_player = df_raw.groupby('username').size()
axes[1].hist(snaps_per_player, bins=30, color='#10b981', alpha=0.8)
axes[1].set_title('Počet měsíčních snapshotů na hráče')
axes[1].set_xlabel('Počet měsíců v historii')
axes[1].set_ylabel('Počet hráčů')

plt.tight_layout()
plt.show()

print(f'\nPrůměrná délka herní historie: {snaps_per_player.mean():.1f} měsíců')
print(f'Hráčů s min. 6 měsíci dat: {(snaps_per_player >= 6).sum():,}')

## 5. Čištění dat (Data Cleaning)

Před trénováním musíme data vyčistit:
- Odstranit duplicitní záznamy (stejný hráč, stejný měsíc)
- Filtrovat hráče s příliš krátkou historií (< 4 měsíce nestačí pro spolehlivé featury)
- Filtrovat outlier Elo záznamy mimo rozsah modelu

In [ ]:
print(f'Před čištěním: {len(df_raw):,} řádků')

# 1. Odstranění duplicit (stejný hráč + time_class + měsíc)
df_clean = df_raw.drop_duplicates(subset=['username', 'time_class', 'month_idx'])
print(f'Po odstranění duplicit: {len(df_clean):,} řádků')

# 2. Filtrování Elo mimo rozsah 600–2000
df_clean = df_clean[(df_clean['elo'] >= 600) & (df_clean['elo'] <= 2000)]
print(f'Po filtrování Elo (600–2000): {len(df_clean):,} řádků')

# 3. Zachování jen hráčů s dostatkem dat (min 4 měsíce)
player_counts = df_clean.groupby(['username', 'time_class']).size()
valid_players = player_counts[player_counts >= 4].index
df_clean = df_clean.set_index(['username', 'time_class']).loc[
    df_clean.set_index(['username', 'time_class']).index.isin(valid_players)
].reset_index()
print(f'Po filtrování hráčů (min. 4 měsíce): {len(df_clean):,} řádků')
print(f'Unikátních hráčů po čištění: {df_clean["username"].nunique():,}')

## 6. Feature Engineering — Tvorba atributů

Surová data (jen Elo a měsíc) nestačí. Musíme z nich vygenerovat smysluplné atributy, které popisují chování hráče:

| Atribut | Popis | Proč je důležitý |
|---|---|---|
| `current_elo` | Aktuální rating | Základní vstup |
| `month_idx` | Věk účtu v měsících | Čerství hráči jiní než veteráni |
| `growth_3m` | Průměrný měsíční nárůst (okno 3M) | Trend — roste či stagnuje? |
| `volatility_3m` | Rozptyl Elo za posl. 3 měsíce | Stabilita hráče |
| `peak_elo` | Historické maximum | Blízkost k vrcholu |
| `elo_vs_peak` | Vzdálenost od maxima | Potenciál k návratu |
| `avg_activity` | Průměrný počet her/měsíc | Aktivita = klíč k růstu |
| `max_gap` | Nejdelší herní pauza (měsíce) | Detektor výpadků formy |
| `time_class_enc` | Kódovaný typ hry | Blitz/Rapid/Bullet mají jiné křivky |

In [ ]:
def encode_tc(tc):
    return {'blitz': 0, 'rapid': 1, 'bullet': 2}.get(tc, 0)

def build_training_pairs(group):
    """Pro jednoho hráče vytvoří seznam (features, target_delta) párů."""
    group = group.drop_duplicates('month_idx').sort_values('month_idx')
    if len(group) < 4:
        return []

    elos   = group['elo'].values
    months = group['month_idx'].values
    counts = group['games_count'].values if 'games_count' in group.columns else np.ones(len(elos))
    peak   = np.max(elos)
    tc_enc = encode_tc(group['time_class'].iloc[0])

    pairs = []
    for i in range(3, len(elos) - 1):
        window_elo = elos[max(0, i-3):i+1]
        current_elo = elos[i]
        # TARGET: o kolik bodů se Elo zvedne příští měsíc
        target_delta = elos[i+1] - current_elo

        window_cnt  = counts[max(0, i-3):i+1]
        avg_activity = np.mean(window_cnt)

        month_diffs = np.diff(months[max(0, i-3):i+1])
        max_gap = int(np.max(month_diffs)) if len(month_diffs) > 0 else 0

        growth_3m   = np.mean(np.diff(window_elo)) if len(window_elo) > 1 else 0
        volatility  = np.std(window_elo)
        elo_vs_peak = current_elo - peak

        features = [
            current_elo,
            int(months[i]),
            growth_3m,
            volatility,
            peak,
            elo_vs_peak,
            tc_enc,
            avg_activity,
            max_gap,
        ]
        pairs.append((features, target_delta))
    return pairs

FEATURE_NAMES = [
    'current_elo', 'month_idx', 'growth_3m', 'volatility_3m',
    'peak_elo', 'elo_vs_peak', 'time_class_enc', 'avg_activity', 'max_gap'
]

# Stavba feature matice
all_X, all_y = [], []
for (user, tc), grp in df_clean.groupby(['username', 'time_class']):
    for feats, delta in build_training_pairs(grp):
        all_X.append(feats)
        all_y.append(delta)

X_all = np.array(all_X)
y_all = np.array(all_y)

print(f'✅ Vygenerováno {len(X_all):,} trénovacích párů (features, target_delta)')
print(f'   Atributů (features): {X_all.shape[1]}')

df_features_preview = pd.DataFrame(X_all, columns=FEATURE_NAMES)
df_features_preview['target_delta'] = y_all
print('\nUkázka feature matice:')
df_features_preview.head()

## 7. Stratifikace — proč trénujeme více modelů?

Začátečník s Elo 700 se zlepšuje úplně jinak než zkušený hráč s Elo 1700. Pokud by existoval jen jeden model pro všechny, predikce by byly průměrné a nepřesné pro každého.

**Řešení:** Rozdělíme hráče do 4 **Elo pásem** (Bands) a trénujeme pro každé pásmo samostatný model:

| Pásmo | Rozsah Elo | Typický hráč |
|---|---|---|
| `low` | 600–1000 | Začátečník — roste rychle |
| `mid_low` | 1000–1400 | Rekreační hráč |
| `mid_high` | 1400–1800 | Pokročilý hráč |
| `high` | 1800–2000 | Expert — roste pomalu |

In [ ]:
ELO_BANDS = [
    (600,  1000, 'low'),
    (1000, 1400, 'mid_low'),
    (1400, 1800, 'mid_high'),
    (1800, 2100, 'high'),
]

def get_elo_band(elo):
    for lo, hi, label in ELO_BANDS:
        if lo <= elo < hi:
            return label
    return 'mid_low'

# Přidáme band sloupec
bands = [get_elo_band(x[0]) for x in all_X]
df_features_preview['elo_band'] = bands

band_counts = df_features_preview['elo_band'].value_counts()
print('Počet trénovacích párů na Elo pásmo:')
print(band_counts)

# Vizualizace
plt.figure(figsize=(8, 4))
band_counts.plot(kind='bar', color=['#ef4444', '#f59e0b', '#10b981', '#6366f1'])
plt.title('Počet trénovacích vzorků na Elo pásmo (Band)')
plt.ylabel('Počet párů')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 8. Trénink modelu — LightGBM

Pro každé Elo pásmo natrénujeme samostatný **LightGBM Gradient Boosting** model.

Model predikuje **měsíční delta** (o kolik bodů se Elo změní příští měsíc), ne absolutní Elo. Predikce na N měsíců vpřed se pak simuluje opakovaným krokem měsíc po měsíci.

In [ ]:
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Trénujeme jeden model pro blitz (ukázka — v produkci se trénuje pro každý band × každý time_class)
results = {}
trained_models = {}

for band_name in ['low', 'mid_low', 'mid_high', 'high']:
    mask = [b == band_name for b in bands]
    X_band = X_all[mask]
    y_band = y_all[mask]

    if len(X_band) < 10:
        print(f'Pásmo {band_name}: nedostatek dat ({len(X_band)} vzorků), přeskočeno.')
        continue

    X_train, X_test, y_train, y_test = train_test_split(
        X_band, y_band, test_size=0.2, random_state=42
    )

    params = {
        'objective': 'regression',
        'metric': 'rmse',
        'learning_rate': 0.05,
        'num_leaves': 31,
        'min_data_in_leaf': max(5, len(X_train) // 20),
        'verbose': -1,
    }

    train_data = lgb.Dataset(X_train, label=y_train)
    model = lgb.train(params, train_data, num_boost_round=300)

    preds = model.predict(X_test)
    rmse = mean_squared_error(y_test, preds, squared=False)
    mae  = mean_absolute_error(y_test, preds)

    trained_models[band_name] = model
    results[band_name] = {'RMSE': round(rmse, 2), 'MAE': round(mae, 2), 'vzorků': len(X_band)}
    print(f'✅ [{band_name}] natrénováno | RMSE: {rmse:.1f} | MAE: {mae:.1f} | vzorků: {len(X_band)}')

## 9. Vyhodnocení modelu (Evaluace)

- **RMSE** (Root Mean Squared Error) — průměrná chyba predikce v Elo bodech
- **MAE** (Mean Absolute Error) — průměrná absolutní odchylka predikce

Nižší hodnota = přesnější model.

In [ ]:
df_results = pd.DataFrame(results).T
print('=== Výsledky modelu po pásmu (Elo Bands) ===')
print(df_results)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df_results['RMSE'].plot(kind='bar', ax=axes[0], color='#ef4444', alpha=0.8)
axes[0].set_title('RMSE podle Elo pásma')
axes[0].set_ylabel('RMSE (Elo body)')
axes[0].axhline(y=df_results['RMSE'].mean(), color='black', linestyle='--', label='průměr')
axes[0].legend()
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)

df_results['MAE'].plot(kind='bar', ax=axes[1], color='#f59e0b', alpha=0.8)
axes[1].set_title('MAE podle Elo pásma')
axes[1].set_ylabel('MAE (Elo body)')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)

plt.tight_layout()
plt.show()

## 10. Feature Importance — Na co se AI dívá?

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, (band_name, model) in enumerate(trained_models.items()):
    importances = model.feature_importance(importance_type='split')
    sorted_idx = np.argsort(importances)[::-1]

    axes[idx].barh(
        [FEATURE_NAMES[i] for i in sorted_idx],
        importances[sorted_idx],
        color='#7c3aed', alpha=0.8
    )
    axes[idx].set_title(f'Feature Importance — pásmo: {band_name}')
    axes[idx].set_xlabel('Počet použití v rozhodovacích stromech')

plt.suptitle('Na co se AI dívá při predikci Elo (Feature Importance)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 11. Srovnání LightGBM vs. K-NN

**Proč jsme nezůstali u jednoduššího algoritmu K-NN (K-Nejbližších sousedů)?**

K-NN hledá 5 hráčů, kteří měli někdy podobné Elo, a udělá průměr jejich vývoje. Nezajímá ho, že vy hrajete 10× aktivněji nebo máte úplně jiné výkyvy formy.

LightGBM naproti tomu buduje stovky **rozhodovacích stromů** — učí se komplexní podmíněné chování: *"Pokud má hráč Elo 1200, hraje 100 her/měsíc a za poslední 3 měsíce stagnuje, jeho budoucí růst bude asi malý."* Tento kontext K-NN neumí zohlednit.

In [ ]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler

# Srovnání na pásmu mid_low
band_name = 'mid_low'
mask = [b == band_name for b in bands]
X_band = X_all[mask]
y_band = y_all[mask]

if len(X_band) > 10:
    X_tr, X_te, y_tr, y_te = train_test_split(X_band, y_band, test_size=0.2, random_state=42)

    # K-NN
    scaler = StandardScaler()
    X_tr_sc = scaler.fit_transform(X_tr)
    X_te_sc = scaler.transform(X_te)
    knn = KNeighborsRegressor(n_neighbors=5)
    knn.fit(X_tr_sc, y_tr)
    knn_rmse = mean_squared_error(y_te, knn.predict(X_te_sc), squared=False)

    # LightGBM
    lgbm_rmse = results[band_name]['RMSE']

    print(f'Srovnání na pásmu "{band_name}" (nižší = lepší):')
    print(f'  K-NN  RMSE: {knn_rmse:.2f} Elo bodů')
    print(f'  LightGBM RMSE: {lgbm_rmse:.2f} Elo bodů')
    zlepšení = (knn_rmse - lgbm_rmse) / knn_rmse * 100
    print(f'  → LightGBM je přesnější o {zlepšení:.1f}%')

    plt.figure(figsize=(6, 4))
    plt.bar(['K-NN', 'LightGBM'], [knn_rmse, lgbm_rmse], color=['#94a3b8', '#7c3aed'])
    plt.title(f'RMSE: K-NN vs LightGBM (pásmo {band_name})')
    plt.ylabel('RMSE — nižší = přesnější')
    plt.show()
else:
    print(f'Pro pásmo {band_name} není dost dat na srovnání.')

## 12. Závěr

Úspěšně jsme prošli kompletním ML pipelines od sběru dat až po evaluaci:

1. ✅ **Data sesbírána** vlastním crawlerem z Chess.com API (1500+ hráčů)
2. ✅ **Data vyčištěna** — duplikáty, outliers, hráči s krátkou historií
3. ✅ **Feature Engineering** — z Elo časové osy jsme odvodili 9 smysluplných atributů
4. ✅ **Stratifikace** — 4 Elo pásma, každé má vlastní natrénovaný model
5. ✅ **LightGBM model natrénován** a vyhodnocen přes RMSE a MAE
6. ✅ **Srovnání s K-NN** dokazuje, proč byl zvolen LightGBM

Modely jsou uloženy do složky `models/` jako `.pkl` soubory a načítá je webová aplikace (`app.py`) při každém uživatelském dotazu.